# 03 — Visualisation & Business Recommendations

**Project:** Gaming Launch Intelligence — Predicting Commercial Success and Optimising Platform & Launch Strategy

**Purpose:** Turn the final SQL analytical outputs into decision-ready visualisations and recommendations.

This notebook is aligned with the **current SQL export layer**. It does not depend on the retired exports such as `genre_attractiveness.csv`, `market_trend.csv`, `launch_simulator.csv`, `marketing_allocation.csv`, or `new_entrant_performance.csv`.

## Current business questions

1. **Which platform should we prioritise?**
2. **Which genre/platform combinations represent the strongest opportunities?**
3. **Which regions and launch windows deserve attention?**
4. **Which scenarios should management prioritise, conditionally validate, or avoid?**

## Output rule

All figures created by this notebook are saved to exactly one location:

```text
Python_Analysis/outputs/figures/
```

No additional results or figure directories are created by this notebook.


## 1. Setup and output paths

In [17]:
from pathlib import Path

# Project root
PROJECT_ROOT = Path("../..").resolve()

# Project directories
PYTHON_OUTPUTS_DIR = PROJECT_ROOT / "Python_Analysis" / "outputs"
FIGURES_DIR = PYTHON_OUTPUTS_DIR / "figures"
SQL_RESULT_DIR = PROJECT_ROOT / "SQL_Analysis" / "Result"

# Create the single agreed figure directory
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("SQL results  :", SQL_RESULT_DIR)
print("Figures      :", FIGURES_DIR)

Project root : C:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence
SQL results  : C:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\SQL_Analysis\Result
Figures      : C:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Python_Analysis\outputs\figures


In [18]:
# ============================================================
# Imports
# ============================================================

import warnings

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

warnings.filterwarnings("ignore")

print("Imports successful.")

Imports successful.


## 2. Load the current SQL exports

The current export layer provides nine decision-support datasets:

- `phase2_market_opportunity.csv`
- `phase2_competitive_accessibility.csv`
- `phase2_genre_platform_fit.csv`
- `phase2_regional_opportunity.csv`
- `phase4_launch_timing_scenarios.csv`
- `phase4_release_competition_density.csv`
- `phase4_platform_lifecycle.csv`
- `phase5_scenario_score.csv`
- `phase5_final_launch_decision.csv`

The loader below fails early if one of these required files is missing. This is intentional: it prevents the notebook from silently using stale analytical outputs.


In [19]:
# ------------------------------------------------------------
# Required SQL exports
# ------------------------------------------------------------

EXPORT_FILES = {
    "market_opportunity": "phase2_market_opportunity.csv",
    "competitive_accessibility": "phase2_competitive_accessibility.csv",
    "genre_platform_fit": "phase2_genre_platform_fit.csv",
    "regional_opportunity": "phase2_regional_opportunity.csv",
    "launch_timing": "phase4_launch_timing_scenarios.csv",
    "release_competition": "phase4_release_competition_density.csv",
    "platform_lifecycle": "phase4_platform_lifecycle.csv",
    "scenario_score": "phase5_scenario_score.csv",
    "final_decision": "phase5_final_launch_decision.csv",
}

missing = [name for name, filename in EXPORT_FILES.items()
           if not (SQL_RESULT_DIR / filename).exists()]

if missing:
    raise FileNotFoundError(
        "Missing required SQL exports:\n"
        + "\n".join(f"  - {EXPORT_FILES[name]}" for name in missing)
        + "\n\nRun the final SQL export file first."
    )

sql_data = {
    name: pd.read_csv(SQL_RESULT_DIR / filename)
    for name, filename in EXPORT_FILES.items()
}

for name, frame in sql_data.items():
    print(f"{name:24s} {len(frame):>8,} rows | {len(frame.columns):>2} columns")


market_opportunity             17 rows | 13 columns
competitive_accessibility       20 rows | 12 columns
genre_platform_fit            237 rows |  8 columns
regional_opportunity           80 rows | 10 columns
launch_timing                  17 rows |  7 columns
release_competition        15,293 rows |  6 columns
platform_lifecycle            281 rows |  7 columns
scenario_score                948 rows | 17 columns
final_decision                948 rows | 17 columns


## 3. Standardise column names and validate the analytical grain

The SQL layer has evolved during the project. Some analytical views use `console`, while others may use `platform`. The helper below standardises only the platform identifier needed by this notebook. It does **not** change the underlying CSVs.

This keeps the visualisation layer resilient without reintroducing the old analytical outputs.


In [20]:
# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def pick_column(df, candidates, required=True):
    """Return the first matching column from a list of candidates."""
    for col in candidates:
        if col in df.columns:
            return col
    if required:
        raise KeyError(
            f"None of these columns were found: {candidates}\n"
            f"Available columns: {list(df.columns)}"
        )
    return None


def numeric(df, col):
    """Convert a column to numeric without mutating the source dataframe."""
    return pd.to_numeric(df[col], errors="coerce")


# ============================================================
# Figure saving
# ============================================================

def save_figure(fig, filename, width=1400, height=800, scale=2):
    """
    Save Plotly figure as PNG.

    All figures are stored in:
    Python_Analysis/outputs/figures/

    Parameters
    ----------
    fig : plotly.graph_objects.Figure
        Plotly figure to save.

    filename : str
        Filename without extension.

    width : int
        Image width in pixels.

    height : int
        Image height in pixels.

    scale : int
        Export resolution multiplier.
    """

    # Ensure .png extension
    if not filename.lower().endswith(".png"):
        filename = f"{filename}.png"

    output_path = FIGURES_DIR / filename

    # Save as PNG
    fig.write_image(
        output_path,
        format="png",
        width=width,
        height=height,
        scale=scale
    )

    print(f"Saved figure: {output_path}")


def clean_text(series):
    return series.astype("string").str.strip()


# Make working copies.
market = sql_data["market_opportunity"].copy()
competitive = sql_data["competitive_accessibility"].copy()
genre_platform = sql_data["genre_platform_fit"].copy()
regional = sql_data["regional_opportunity"].copy()
timing = sql_data["launch_timing"].copy()
competition = sql_data["release_competition"].copy()
lifecycle = sql_data["platform_lifecycle"].copy()
scenario = sql_data["scenario_score"].copy()
decision = sql_data["final_decision"].copy()

# Platform aliases used by different SQL layers.
for frame in [genre_platform, lifecycle, scenario, decision]:
    platform_col = pick_column(
        frame,
        ["platform", "console", "platform_name", "platform_family"],
        required=False
    )
    if platform_col and platform_col != "platform":
        frame["platform"] = frame[platform_col].astype("string")

print("\nLoaded analytical tables successfully.")



Loaded analytical tables successfully.


## Q1 — Which platform should we prioritise?

We use the current platform lifecycle output together with the final strategic scenario score.

The goal is not to declare a platform universally best. Instead, we identify platforms that combine:

- sustained historical presence
- strong strategic scenario scores
- enough evidence to justify prioritisation


In [21]:
print("Platform lifecycle columns:")
print(lifecycle.columns.tolist())

print("\nFirst 5 rows:")
display(lifecycle.head())

Platform lifecycle columns:
['platform', 'release_year', 'releases', 'first_observed_year', 'last_observed_year', 'lifecycle_position', 'lifecycle_stage']

First 5 rows:


,platform,release_year,releases,first_observed_year,last_observed_year,lifecycle_position,lifecycle_stage
0,2600,1977,3,1977,1990,0.000000,EARLY
1,2600,1978,7,1977,1990,0.076923,EARLY
2,2600,1979,1,1977,1990,0.153846,EARLY
3,2600,1980,5,1977,1990,0.230769,GROWTH
4,2600,1981,6,1977,1990,0.307692,GROWTH


In [22]:
# ------------------------------------------------------------
# Q1A. Platform lifecycle heatmap
# ------------------------------------------------------------

# Ensure correct data types
lifecycle["release_year"] = pd.to_numeric(
    lifecycle["release_year"],
    errors="coerce"
)

lifecycle["releases"] = pd.to_numeric(
    lifecycle["releases"],
    errors="coerce"
)

# Create platform × year matrix using release volume
life_pivot = (
    lifecycle
    .dropna(subset=["platform", "release_year", "releases"])
    .pivot_table(
        index="platform",
        columns="release_year",
        values="releases",
        aggfunc="sum",
        fill_value=0
    )
)

# Keep platforms with meaningful historical presence
platform_totals = life_pivot.sum(axis=1)

life_pivot = life_pivot.loc[
    platform_totals[platform_totals > 1].index
]

# Plot
fig = px.imshow(
    life_pivot,
    aspect="auto",
    labels={
        "x": "Release Year",
        "y": "Platform",
        "color": "Number of Releases"
    },
    title="Platform Lifecycle: Historical Release Activity"
)

fig.update_layout(
    height=max(450, 25 * len(life_pivot))
)

save_figure(
    fig,
    "platform_lifecycle_heatmap"
)

fig.show()

Saved figure: C:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Python_Analysis\outputs\figures\platform_lifecycle_heatmap.png


In [23]:
# ------------------------------------------------------------
# Q1B. Platform strategic score
# ------------------------------------------------------------

scenario_platform = (
    scenario.dropna(subset=["platform"])
    .assign(strategic_scenario_score=lambda x: numeric(x, "strategic_scenario_score"))
    .groupby("platform", as_index=False)
    .agg(
        avg_strategic_score=("strategic_scenario_score", "mean"),
        best_strategic_score=("strategic_scenario_score", "max"),
        scenarios=("strategic_scenario_score", "count")
    )
    .sort_values("avg_strategic_score", ascending=False)
)

top_platforms = scenario_platform.head(10)

fig = px.bar(
    top_platforms.sort_values("avg_strategic_score"),
    x="avg_strategic_score",
    y="platform",
    orientation="h",
    text="avg_strategic_score",
    hover_data=["best_strategic_score", "scenarios"],
    labels={
        "avg_strategic_score": "Average Strategic Scenario Score",
        "platform": "Platform"
    },
    title="Platform Prioritisation: Average Strategic Scenario Score"
)
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.update_layout(height=500)
save_figure(fig, "platform_strategic_score")
fig.show()

print("Top platforms by average strategic scenario score:")
display(top_platforms)


Saved figure: C:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Python_Analysis\outputs\figures\platform_strategic_score.png


Top platforms by average strategic scenario score:


,platform,avg_strategic_score,best_strategic_score,scenarios
8,NES,79.320000,86.82,4
0,2600,75.700000,85.38,12
12,PS,74.148333,83.83,48
14,PS3,73.412143,87.98,56
22,X360,73.196000,89.17,60
7,N64,72.250937,82.93,32
13,PS2,72.167500,87.71,48
19,SNES,71.170000,85.04,28
4,GB,70.880000,75.88,4
20,WII,67.787308,80.41,52


### Platform decision logic

A platform should be considered a priority when it repeatedly appears in high-scoring scenarios, rather than because of one isolated title or one historical peak year.

Use the chart above together with the lifecycle heatmap to distinguish:

- **strategic strength:** high scenario scores
- **historical presence:** meaningful release share
- **lifecycle risk:** declining or very short-lived platform presence


## Q2 — Which genre/platform combinations represent the strongest opportunities?

This section uses the current Phase 2 genre/platform fit output and Phase 5 scenario scores.

It replaces the old `genre_attractiveness.csv` and `launch_simulator.csv` framework.


In [24]:
# ------------------------------------------------------------
# Q2A. Genre × platform fit
# ------------------------------------------------------------

gp_score_col = pick_column(
    genre_platform,
    [
        "genre_platform_fit_score",
        "platform_fit_score",
        "median_sales_per_release",
        "avg_sales_per_release",
    ]
)

genre_col = pick_column(genre_platform, ["genre"])
genre_platform["fit_score"] = numeric(genre_platform, gp_score_col)

gp_pivot = (
    genre_platform
    .dropna(subset=[genre_col, "platform", "fit_score"])
    .pivot_table(
        index=genre_col,
        columns="platform",
        values="fit_score",
        aggfunc="mean"
    )
)

# Keep the most decision-relevant genres.
genre_rank = gp_pivot.max(axis=1).sort_values(ascending=False)
gp_pivot = gp_pivot.loc[genre_rank.head(15).index]

fig = px.imshow(
    gp_pivot,
    aspect="auto",
    text_auto=".1f",
    labels={"x": "Platform", "y": "Genre", "color": "Fit / Performance Score"},
    title="Genre × Platform Fit: Highest-Opportunity Combinations"
)
fig.update_layout(height=max(500, 28 * len(gp_pivot)))
save_figure(fig, "genre_platform_fit_heatmap")
fig.show()


Saved figure: C:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Python_Analysis\outputs\figures\genre_platform_fit_heatmap.png


In [25]:
# ------------------------------------------------------------
# Q2B. Market opportunity by genre
# ------------------------------------------------------------

market_score_col = pick_column(
    market,
    [
        "market_opportunity_score",
        "opportunity_score",
        "attractiveness_score",
    ]
)

market["market_score"] = numeric(market, market_score_col)
market_genre_col = pick_column(market, ["genre"])

market_rank = (
    market.dropna(subset=[market_genre_col, "market_score"])
    .groupby(market_genre_col, as_index=False)["market_score"]
    .mean()
    .sort_values("market_score", ascending=False)
    .head(12)
)

fig = px.bar(
    market_rank.sort_values("market_score"),
    x="market_score",
    y=market_genre_col,
    orientation="h",
    text="market_score",
    labels={
        "market_score": "Market Opportunity Score",
        market_genre_col: "Genre"
    },
    title="Top Genre Market Opportunities"
)
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.update_layout(height=500)
save_figure(fig, "genre_market_opportunity")
fig.show()

print("Top genre opportunities:")
display(market_rank)


Saved figure: C:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Python_Analysis\outputs\figures\genre_market_opportunity.png


Top genre opportunities:


,genre,market_score
1,Action-Adventure,79.64
3,Fighting,75.84
14,Sports,75.45
12,Shooter,72.98
8,Platform,70.91
0,Action,64.73
4,Misc,60.94
16,Visual Novel,60.66
7,Party,59.85
10,Racing,59.18


In [26]:
# ------------------------------------------------------------
# Q2C. Final strategic scenario map
# ------------------------------------------------------------

scenario["strategic_scenario_score"] = numeric(
    scenario, "strategic_scenario_score"
)

scenario_genre = pick_column(scenario, ["genre"])
scenario_platform_col = pick_column(scenario, ["platform", "console"])

scenario_plot = (
    scenario
    .dropna(subset=[scenario_genre, scenario_platform_col, "strategic_scenario_score"])
    .copy()
)

# One point per genre-platform pair.
scenario_gp = (
    scenario_plot
    .groupby([scenario_genre, scenario_platform_col], as_index=False)
    .agg(
        strategic_scenario_score=("strategic_scenario_score", "mean")
    )
)

top_scenarios = scenario_gp.nlargest(20, "strategic_scenario_score")

fig = px.scatter(
    top_scenarios,
    x=scenario_platform_col,
    y="strategic_scenario_score",
    color=scenario_genre,
    hover_data=[scenario_genre, scenario_platform_col, "strategic_scenario_score"],
    labels={
        scenario_platform_col: "Platform",
        "strategic_scenario_score": "Strategic Scenario Score",
        scenario_genre: "Genre"
    },
    title="Top Genre × Platform Strategic Scenarios"
)
fig.update_layout(height=600)
save_figure(fig, "top_genre_platform_scenarios")
fig.show()

display(
    top_scenarios.rename(
        columns={
            scenario_genre: "Genre",
            scenario_platform_col: "Platform",
            "strategic_scenario_score": "Strategic Score"
        }
    ).reset_index(drop=True)
)


Saved figure: C:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Python_Analysis\outputs\figures\top_genre_platform_scenarios.png


,Genre,Platform,Strategic Score
0,Action-Adventure,X360,84.170
1,Sports,PS4,83.910
2,Fighting,PS3,82.410
3,Sports,PS2,81.660
4,Sports,PS3,81.410
5,Shooter,XONE,81.380
6,Action-Adventure,PS3,81.050
7,Platform,PS3,80.480
8,Shooter,2600,80.380
9,Fighting,X360,80.300


## Q3 — Which regions and launch windows deserve attention?

The current SQL pipeline provides regional opportunity and launch-timing outputs.

We do **not** recreate the retired `marketing_allocation.csv` or `regional_correlation.csv` outputs. Recommendations are based only on the current exported evidence.


In [27]:
# ------------------------------------------------------------
# Q3A. Regional opportunity
# ------------------------------------------------------------

regional_score_col = pick_column(
    regional,
    [
        "regional_opportunity_score",
        "opportunity_score",
        "regional_score",
        "skew_pct",
    ]
)
regional["regional_score"] = numeric(regional, regional_score_col)

region_col = pick_column(regional, ["region"])
regional_genre_col = pick_column(regional, ["genre"], required=False)

if regional_genre_col:
    region_summary = (
        regional.dropna(subset=[region_col, "regional_score"])
        .groupby(region_col, as_index=False)
        .agg(
            avg_regional_score=("regional_score", "mean"),
            best_regional_score=("regional_score", "max")
        )
        .sort_values("avg_regional_score", ascending=False)
    )
else:
    region_summary = (
        regional.dropna(subset=[region_col, "regional_score"])
        .groupby(region_col, as_index=False)["regional_score"]
        .mean()
        .rename(columns={"regional_score": "avg_regional_score"})
        .sort_values("avg_regional_score", ascending=False)
    )

fig = px.bar(
    region_summary.sort_values("avg_regional_score"),
    x="avg_regional_score",
    y=region_col,
    orientation="h",
    text="avg_regional_score",
    labels={
        "avg_regional_score": "Average Regional Opportunity Score",
        region_col: "Region"
    },
    title="Regional Opportunity Ranking"
)
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.update_layout(height=450)
save_figure(fig, "regional_opportunity")
fig.show()

display(region_summary)


Saved figure: C:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Python_Analysis\outputs\figures\regional_opportunity.png


,region,avg_regional_score,best_regional_score
1,North America,69.167,100.00
3,PAL,55.000,100.00
0,Japan,50.001,100.00
2,Other,25.834,66.67


In [29]:
# ------------------------------------------------------------
# Q3B. Launch timing opportunity
# ------------------------------------------------------------

# Identify the historical timing score
timing_score_col = pick_column(
    timing,
    [
        "historical_timing_score",
        "timing_score",
        "opportunity_score"
    ]
)

timing["timing_score"] = numeric(
    timing,
    timing_score_col
)


# ------------------------------------------------------------
# Recommended historical launch month
# ------------------------------------------------------------

month_col = pick_column(
    timing,
    [
        "recommended_historical_month",
        "release_month",
        "month"
    ]
)


# ------------------------------------------------------------
# Genre availability
# ------------------------------------------------------------

timing_genre_col = pick_column(
    timing,
    ["genre"],
    required=False
)


# ------------------------------------------------------------
# Prepare plotting data
# ------------------------------------------------------------

if timing_genre_col:

    timing_plot = (
        timing
        .dropna(subset=[month_col, "timing_score"])
        .groupby(
            [timing_genre_col, month_col],
            as_index=False
        )
        .agg(
            timing_score=("timing_score", "mean")
        )
    )

    # Keep the strongest genres for readability
    top_timing_genres = (
        timing_plot
        .groupby(timing_genre_col)["timing_score"]
        .mean()
        .nlargest(8)
        .index
    )

    timing_plot = timing_plot[
        timing_plot[timing_genre_col].isin(top_timing_genres)
    ]

else:

    timing_plot = (
        timing
        .dropna(subset=[month_col, "timing_score"])
        .groupby(
            month_col,
            as_index=False
        )
        .agg(
            timing_score=("timing_score", "mean")
        )
    )


# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------

fig = px.bar(
    timing_plot.sort_values(
        "timing_score",
        ascending=False
    ).head(20),

    x=month_col,
    y="timing_score",

    color=(
        timing_genre_col
        if timing_genre_col
        else None
    ),

    labels={
        month_col: "Recommended Historical Launch Month",
        "timing_score": "Historical Timing Score"
    },

    title="Historically Attractive Launch Timing"
)

fig.update_layout(
    height=550
)

save_figure(
    fig,
    "launch_timing_opportunity"
)

fig.show()

Saved figure: C:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Python_Analysis\outputs\figures\launch_timing_opportunity.png


In [30]:
# ------------------------------------------------------------
# Q3C. Competitive release density
# ------------------------------------------------------------

density_col = pick_column(
    competition,
    [
        "comparable_releases_30d",
        "release_count_30d",
        "competition_density",
        "comparable_release_count"
    ]
)

competition["release_density"] = numeric(competition, density_col)

comp_dimension = pick_column(
    competition,
    ["genre", "console", "platform"],
    required=False
)

if comp_dimension:
    density_summary = (
        competition.dropna(subset=["release_density"])
        .groupby(comp_dimension, as_index=False)
        .agg(
            avg_competition=("release_density", "mean"),
            peak_competition=("release_density", "max")
        )
        .sort_values("avg_competition", ascending=False)
        .head(15)
    )

    fig = px.bar(
        density_summary.sort_values("avg_competition"),
        x="avg_competition",
        y=comp_dimension,
        orientation="h",
        text="avg_competition",
        hover_data=["peak_competition"],
        labels={
            "avg_competition": "Average Comparable Releases in 30-Day Window",
            comp_dimension: comp_dimension.replace("_", " ").title()
        },
        title="Competitive Release Density"
    )
    fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
    fig.update_layout(height=550)
    save_figure(fig, "release_competition_density")
    fig.show()

    display(density_summary)
else:
    print("No suitable competition dimension was found; displaying summary statistics only.")
    display(competition["release_density"].describe())


Saved figure: C:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Python_Analysis\outputs\figures\release_competition_density.png


,genre,avg_competition,peak_competition
6,Misc,40.999406,444
0,Action,38.253589,360
17,Sports,36.864343,432
2,Adventure,26.926561,315
12,Racing,16.330097,138
16,Simulation,14.891732,108
15,Shooter,14.846999,160
10,Platform,14.014045,116
13,Role-Playing,12.224383,108
1,Action-Adventure,11.516340,70


## Q4 — Which scenarios should management prioritise?

This is the final decision layer.

The current SQL pipeline classifies scenarios as:

- **GO**
- **CONDITIONAL**
- **AVOID**

The notebook does not recreate the old launch simulator. It consumes the final decision view directly.


In [31]:
# ------------------------------------------------------------
# Q4A. Decision distribution
# ------------------------------------------------------------

decision_col = pick_column(decision, ["final_decision", "decision"])
decision[decision_col] = clean_text(decision[decision_col])

decision_counts = (
    decision[decision_col]
    .value_counts(dropna=False)
    .rename_axis("decision")
    .reset_index(name="scenario_count")
)

fig = px.bar(
    decision_counts,
    x="decision",
    y="scenario_count",
    text="scenario_count",
    labels={"decision": "Final Decision", "scenario_count": "Scenarios"},
    title="Final Launch Decision Distribution"
)
fig.update_traces(textposition="outside")
fig.update_layout(height=450)
save_figure(fig, "final_decision_distribution")
fig.show()

display(decision_counts)


Saved figure: C:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Python_Analysis\outputs\figures\final_decision_distribution.png


,decision,scenario_count
0,CONDITIONAL,929
1,AVOID,10
2,GO,9


In [32]:
# ------------------------------------------------------------
# Q4B. Top recommended scenarios
# ------------------------------------------------------------

decision["strategic_scenario_score"] = numeric(
    decision, "strategic_scenario_score"
)

score_col = "strategic_scenario_score"

decision_order = {"GO": 1, "CONDITIONAL": 2, "AVOID": 3}
decision["_decision_order"] = decision[decision_col].map(decision_order).fillna(4)

top_decisions = (
    decision
    .sort_values(
        ["_decision_order", score_col],
        ascending=[True, False]
    )
    .head(15)
    .copy()
)

# Show the most useful decision columns when available.
preferred_cols = [
    "genre",
    "platform",
    "console",
    "region",
    "release_month",
    "strategic_scenario_score",
    "success_probability",
    "decision_confidence",
    "final_decision",
    "primary_opportunity",
    "primary_risk",
    "recommended_action",
]

display_cols = [c for c in preferred_cols if c in top_decisions.columns]

top_decisions_display = top_decisions[display_cols].reset_index(drop=True)
display(top_decisions_display)


,genre,platform,region,strategic_scenario_score,success_probability,decision_confidence,final_decision,primary_opportunity,primary_risk,recommended_action
0,Sports,PS3,North America,86.41,0.755913,HIGH,GO,Strong market opportunity,Execution risk requires validation,Proceed to commercial planning and detailed fi...
1,Sports,X360,North America,83.91,0.787068,HIGH,GO,Strong market opportunity,Execution risk requires validation,Proceed to commercial planning and detailed fi...
2,Sports,PS3,Other,83.91,0.755913,HIGH,GO,Strong market opportunity,Execution risk requires validation,Proceed to commercial planning and detailed fi...
3,Sports,X360,Other,81.41,0.787068,HIGH,GO,Strong market opportunity,Execution risk requires validation,Proceed to commercial planning and detailed fi...
4,Sports,PS3,PAL,81.41,0.755913,HIGH,GO,Strong market opportunity,Execution risk requires validation,Proceed to commercial planning and detailed fi...
5,Sports,X360,PAL,78.91,0.787068,HIGH,GO,Strong market opportunity,Execution risk requires validation,Proceed to commercial planning and detailed fi...
6,Music,X360,PAL,75.40,0.663012,HIGH,GO,Strong historical platform fit,Execution risk requires validation,Proceed to commercial planning and detailed fi...
7,Music,X360,Japan,75.40,0.663012,HIGH,GO,Strong historical platform fit,Execution risk requires validation,Proceed to commercial planning and detailed fi...
8,Music,X360,North America,75.40,0.663012,HIGH,GO,Strong historical platform fit,Execution risk requires validation,Proceed to commercial planning and detailed fi...
9,Action-Adventure,X360,PAL,89.17,NaN,MEDIUM,CONDITIONAL,Strong market opportunity,Low predicted commercial success,Run the pre-launch model and validate the scen...


In [33]:
# ------------------------------------------------------------
# Q4C. Final scenario score distribution by decision
# ------------------------------------------------------------

score_dist = decision.dropna(subset=[score_col]).copy()

fig = px.box(
    score_dist,
    x=decision_col,
    y=score_col,
    points=False,
    labels={
        decision_col: "Final Decision",
        score_col: "Strategic Scenario Score"
    },
    title="Strategic Scenario Score by Final Decision"
)
fig.update_layout(height=500)
save_figure(fig, "scenario_score_by_decision")
fig.show()


Saved figure: C:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Python_Analysis\outputs\figures\scenario_score_by_decision.png


## 5. Executive recommendation generator

The statements below are generated directly from the current SQL outputs. They are deliberately framed as **historical evidence and decision support**, not as guarantees of future commercial performance.


In [ ]:
# ------------------------------------------------------------
# Executive recommendation summary
# ------------------------------------------------------------

print("=" * 80)
print("EXECUTIVE DECISION SUMMARY")
print("=" * 80)

# 1. Platform
if not scenario_platform.empty:
    best_platform = scenario_platform.iloc[0]
    print(
        f"\n1. PLATFORM PRIORITY\n"
        f"   {best_platform['platform']} has the highest average strategic "
        f"scenario score ({best_platform['avg_strategic_score']:.1f}) "
        f"across {int(best_platform['scenarios'])} evaluated scenarios."
    )

# 2. Genre
if not market_rank.empty:
    best_genre = market_rank.iloc[0]
    print(
        f"\n2. GENRE OPPORTUNITY\n"
        f"   {best_genre[market_genre_col]} ranks highest on the current "
        f"market opportunity score ({best_genre['market_score']:.1f})."
    )

# 3. Region
if not region_summary.empty:
    best_region = region_summary.iloc[0]
    print(
        f"\n3. REGIONAL PRIORITY\n"
        f"   {best_region[region_col]} has the highest average regional "
        f"opportunity score ({best_region['avg_regional_score']:.1f})."
    )

# 4. Timing
if not timing_plot.empty:
    best_timing = timing_plot.sort_values("timing_score", ascending=False).iloc[0]
    timing_label = (
        f"{best_timing[month_col]}"
        + (
            f" / {best_timing[timing_genre_col]}"
            if timing_genre_col and timing_genre_col in timing_plot.columns
            else ""
        )
    )
    print(
        f"\n4. TIMING OPPORTUNITY\n"
        f"   The strongest observed timing scenario is {timing_label}, "
        f"with a historical timing score of {best_timing['timing_score']:.1f}."
    )

# 5. Final decision
decision_counts_map = decision_counts.set_index("decision")["scenario_count"].to_dict()
print(
    f"\n5. FINAL DECISION MIX\n"
    f"   GO: {int(decision_counts_map.get('GO', 0)):,}\n"
    f"   CONDITIONAL: {int(decision_counts_map.get('CONDITIONAL', 0)):,}\n"
    f"   AVOID: {int(decision_counts_map.get('AVOID', 0)):,}"
)

print("\nIMPORTANT:")
print("These recommendations summarise historical evidence in the current SQL outputs.")
print("They should be combined with financial modelling, current market conditions,")
print("product strategy, and management judgement before an actual launch decision.")


EXECUTIVE DECISION SUMMARY

1. PLATFORM PRIORITY
   NES has the highest average strategic scenario score (79.3) across 4 evaluated scenarios.

2. GENRE OPPORTUNITY
   Action-Adventure ranks highest on the current market opportunity score (79.6).

3. REGIONAL PRIORITY
   North America has the highest average regional opportunity score (69.2).

4. TIMING OPPORTUNITY
   The strongest observed timing scenario is 10 / Misc, with a historical timing score of 100.0.

5. FINAL DECISION MIX
   GO: 9
   CONDITIONAL: 929
   AVOID: 10

IMPORTANT:
These recommendations summarise historical evidence in the current SQL outputs.
They should be combined with financial modelling, current market conditions,
product strategy, and management judgement before an actual launch decision.


## 6. Final output inventory

All visualisations from this notebook are written to:

```text
Python_Analysis/outputs/figures/
```

The notebook does **not** create another results directory.

The SQL datasets consumed by this notebook remain in:

```text
SQL_Analysis/Result/
```

### Final analytical flow

```text
02_statistical_analysis.ipynb
        ↓
Python_Analysis/outputs/
        ↓
PostgreSQL Phase 2–5
        ↓
SQL_Analysis/Result/
        ↓
03_visualisation_and_recommendations.ipynb
        ↓
Python_Analysis/outputs/figures/
        ↓
Power BI
```

This keeps Python analytical outputs, SQL decision-support exports, and visual artefacts separated while maintaining a single source of truth for each layer.
